# Jupyter Macromolecule Visualization

This notebook shows the notebook-native path for protein assemblies, membrane systems, ATP/ADP pockets, and MD trajectories. It avoids desktop molecular viewers.

Use it with:

```bash
uv sync --extra notebook --extra viz
uv run jupyter lab notebooks/macromolecule-viz
```

Tool split:

- `py3Dmol`: static PDB/mmCIF structures, binding-pocket views, surfaces, labels, simple distance marks.
- `Plotly`: preloaded trajectory preview with a play button and frame slider.
- `MDAnalysis` / `MDTraj`: trajectory loading and numeric analysis.
- `ProLIF`: ligand-protein interaction fingerprints across frames.
- `Plotly`: time traces for contacts, distances, and reaction coordinates.


In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown

NOTEBOOK_DIR = Path("notebooks/macromolecule-viz") if Path("notebooks/macromolecule-viz").exists() else Path(".")
if str(NOTEBOOK_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

from helpers import analysis, config as nb_config, static_views, trajectory_preview, workflow

analysis = importlib.reload(analysis)
nb_config = importlib.reload(nb_config)
static_views = importlib.reload(static_views)
trajectory_preview = importlib.reload(trajectory_preview)
workflow = importlib.reload(workflow)


## Static Structure: ATP-Bound Receptor

`py3Dmol` is the fastest notebook viewer for a static structure. The default example opens a real ATP-bound receptor structure from RCSB PDB 4DW1: the ATP-gated P2X4 ion channel. Replace the path with your own membrane-protein assembly or point `load_structure_view` at a local `.pdb` / `.cif` file. The viewer width is responsive to the notebook cell; only the height is fixed so the molecule has stable vertical space.


In [ ]:
paths = nb_config.notebook_paths()
protocol = nb_config.MDProtocol()
settings = nb_config.PreviewSettings()

DATA_DIR = paths.data_dir
ATP_RECEPTOR_PDB = paths.atp_receptor_pdb
ATP_LIGAND_SDF = paths.atp_ligand_sdf
PREPARED_DIR = paths.prepared_dir
PREPARED_TRAJECTORY = paths.prepared_trajectory
RECEPTOR_SELECTION = nb_config.RECEPTOR_SELECTION
LIGAND_SELECTION = nb_config.LIGAND_SELECTION
VIEWER_WIDTH = settings.viewer_width
VIEWER_HEIGHT = settings.viewer_height
RUN_MLX_ON_THE_FLY = True

load_structure_view = static_views.load_structure_view
load_structure_view(ATP_RECEPTOR_PDB, width=VIEWER_WIDTH, height=VIEWER_HEIGHT)


## Local Structure Files

This cell opens a real 3D ATP SDF from PubChem CID 5957, stored under `notebooks/macromolecule-viz/data/`. Use SDF for ligand-only inspection. Use PDB/mmCIF plus prepared topology/trajectory files for receptor interaction analysis and MD.


In [ ]:
load_structure_view(ATP_LIGAND_SDF)


## Notebook-Native MLX MD Artifact

For the bundled 4DW1 ATP-pocket example, this notebook now builds the internal all-atom production MLX artifact directly from the shipped structure data when needed. It overwrites stale generated demo artifacts, runs MLX minimization + restrained NVT + production NVT if the trajectory is missing or stale, then visualizes and analyzes the saved coordinates.

This is fixed-topology classical MD: no ATP hydrolysis, no bond breaking, and no docking/search. Unsupported user systems still need real topology/parameter import and fail with explicit errors.


In [ ]:
production_result = workflow.ensure_production_trajectory(
    paths,
    protocol,
    settings,
    run_mlx_on_the_fly=RUN_MLX_ON_THE_FLY,
)

real_trajectory_loaded = production_result.real_trajectory_loaded
trajectory_source = production_result.trajectory_source
u = production_result.universe
prepared_bundle = production_result.prepared_bundle
prepared_artifact = production_result.prepared_artifact
mlx_trajectory_record = production_result.trajectory_record
production_error = production_result.production_error

display(Markdown(workflow.production_status_markdown(production_result, paths, protocol)))


## Production MLX Trajectory Animation

This cell only displays a trajectory when `trajectory.npz` was generated by `mlx_atomistic` from a production-valid artifact. Coordinates are shown as stored; no display-only motion amplification is applied.


In [ ]:
if not real_trajectory_loaded:
    display(Markdown("**Skipped.** No production MLX trajectory is loaded."))
else:
    preview = trajectory_preview.build_trajectory_preview(
        prepared_artifact,
        mlx_trajectory_record,
        viewer_height=settings.viewer_height,
        inspection_cutoff_angstrom=settings.inspection_cutoff_angstrom,
        play_interval_ms=settings.trajectory_play_interval_ms,
        original_ligand_ghost_color=settings.original_ligand_ghost_color,
    )

    display(preview.playback_df)
    display(preview.controls_df)
    display(preview.element_df)

    display(Markdown("**ATP center-of-mass translation.** This metric separates real ligand translation from rocking/stretching."))
    display(preview.ligand_motion_summary_df)
    preview.ligand_motion_figure.show()

    display(Markdown("**Preloaded Plotly trajectory preview.** The Play button and slider control stored MLX frames directly; this avoids NGLView comm-streaming jitter."))
    preview.trajectory_figure.show()


## Production Interaction Analysis

These panels are generated only for a production MLX trajectory. They report real trajectory diagnostics and geometric interaction observables; hydrogen-bond and ProLIF analysis stay disabled unless explicit hydrogens and complete topology are present.


In [ ]:
if not real_trajectory_loaded:
    display(Markdown("**Skipped.** Production interaction analysis requires a valid MLX trajectory."))
else:
    diagnostics_df = analysis.diagnostics_dataframe(mlx_trajectory_record)
    display(diagnostics_df.head())
    analysis.diagnostics_figure(diagnostics_df).show()

    stride = max(1, len(u.trajectory) // 250)
    interaction_df = analysis.atp_receptor_interaction_trace(
        u,
        ligand_selection=LIGAND_SELECTION,
        receptor_selection=RECEPTOR_SELECTION,
        stride=stride,
    )
    rmsd_df = analysis.atp_rmsd(u, ligand_selection=LIGAND_SELECTION)
    rmsf_df = analysis.pocket_rmsf(u, receptor_selection=RECEPTOR_SELECTION)

    display(interaction_df.head())
    if len(interaction_df) > 1:
        analysis.interaction_figure(interaction_df).show()
    if not rmsd_df.empty:
        display(rmsd_df.head())
        analysis.rmsd_figure(rmsd_df).show()
    if not rmsf_df.empty:
        display(rmsf_df.sort_values("rmsf_A", ascending=False).head(20))


## Hydrogen Bonds and ProLIF

Hydrogen-bond and fingerprint analysis are only meaningful for chemically complete all-atom topology. This cell stays fail-closed when hydrogens or ligand topology are incomplete.


In [ ]:
if not real_trajectory_loaded:
    display(Markdown("**Skipped.** Hydrogen-bond and ProLIF analysis require a production MLX trajectory."))
else:
    symbols = np.asarray(prepared_artifact.symbols).astype(str)
    hydrogen_count = int(np.count_nonzero(np.char.upper(symbols) == "H"))
    ligand = u.select_atoms(LIGAND_SELECTION)
    protein = u.select_atoms(RECEPTOR_SELECTION)
    if hydrogen_count == 0:
        display(Markdown("**Skipped.** No explicit hydrogens are present."))
    elif len(ligand) == 0 or len(protein) == 0:
        display(Markdown("**Skipped.** Ligand or protein selection is empty."))
    else:
        hbond_df, hbond_error = analysis.run_hydrogen_bond_analysis(u)
        if hbond_error is not None:
            print("Hydrogen-bond analysis needs complete donor/hydrogen/acceptor topology.")
            print(f"Original error: {hbond_error}")
        else:
            display(hbond_df.head())
            if not hbond_df.empty:
                display(hbond_df.groupby("acceptor_index").size().sort_values(ascending=False).head(20))

        prolif_df, prolif_error = analysis.run_prolif_fingerprint(u, ligand, protein)
        if prolif_error is not None:
            print("ProLIF needs chemically complete ligand/protein topology and bond perception.")
            print(f"Original error: {prolif_error}")
        else:
            display(prolif_df.head())
            display(prolif_df.mean().sort_values(ascending=False).head(20))


## What to Look At for ATP Reactions

For ATP binding, visualize the ligand pocket and contact occupancy. For ATP chemistry, track mechanism-specific distances and angles rather than relying on a static scene.

Useful coordinates:

- ATP/ADP phosphates to catalytic residues.
- Mg2+ coordination to beta/gamma phosphate oxygens.
- Nucleophile to P_gamma distance for phosphoryl transfer.
- Leaving-group P-O distance during hydrolysis.
- Pocket opening/closing distances across the protein assembly.
- Lipid or membrane-contact occupancy for membrane-coupled conformational states.
